In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import timm
from tqdm.notebook import tqdm
import numpy as np
from sklearn.model_selection import StratifiedKFold

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {device}")

Menggunakan device: cuda:0


### Persiapan Dataset & K-Fold (Tanpa Pseudo-Label)

In [2]:
BATCH_SIZE = 16 
EPOCHS = 12     
LEARNING_RATE = 1e-4
IMG_SIZE = 224
K_FOLDS = 5

data_dir = '../Data/train_cropped/' 

# Augmentasi dengan tambahan sedikit Noise/Blur untuk simulasi kamera jelek
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# buat 2 instance ImageFolder agar transformasinya tidak bentrok
dataset_train_base = datasets.ImageFolder(root=data_dir, transform=train_transform)
dataset_val_base = datasets.ImageFolder(root=data_dir, transform=val_transform)

class_names = dataset_train_base.classes
targets = dataset_train_base.targets 

# Siapkan Stratified K-Fold (agar sebaran kelas palsu dan asli tetap seimbang di setiap lipatan)
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

model_dir = '../Models/'
os.makedirs(model_dir, exist_ok=True)

### The K-Fold Training Loop

In [3]:
fold_results = {}

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(targets)), targets)):
    print(f"\n{'='*40}")
    print(f"MEMULAI FOLD {fold + 1}/{K_FOLDS}")
    print(f"{'='*40}")
    
    # Bungkus index ke dalam Subset
    train_sub = Subset(dataset_train_base, train_idx)
    val_sub = Subset(dataset_val_base, val_idx)
    
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_sub, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # INISIALISASI MODEL BARU UNTUK SETIAP FOLD!
    model = timm.create_model('convnext_tiny', pretrained=True, num_classes=len(class_names), drop_rate=0.3)
    model = model.to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    
    best_val_acc = 0.0
    patience = 3
    patience_counter = 0
    model_save_path = os.path.join(model_dir, f'convnext_fold_{fold+1}.pth')
    
    for epoch in range(EPOCHS):
        # --- TRAIN ---
        model.train()
        running_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f'Fold {fold+1} - Epoch {epoch+1}/{EPOCHS} [Train]', leave=False)
        
        for inputs, labels in train_pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * inputs.size(0)
            
        scheduler.step()
        epoch_train_loss = running_loss / len(train_sub)
        
        # --- VAL ---
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_sub)
        epoch_val_acc = correct / total
        
        print(f"Fold {fold+1} | Epoch {epoch+1} | Val Acc: {epoch_val_acc:.4f} | Val Loss: {epoch_val_loss:.4f}")
        
        # --- SAVE BEST MODEL PER FOLD ---
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), model_save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"Early Stopping di Fold {fold+1} pada Epoch {epoch+1}")
            break
            
    fold_results[f'Fold {fold+1}'] = best_val_acc
    print(f"Fold {fold+1} Selesai! Model terbaik disimpan di {model_save_path} dengan Akurasi: {best_val_acc:.4f}")

print("\n" + "="*40)
print("SEMUA FOLD SELESAI DILATIH!")
print("Ringkasan Akurasi Validasi:")
for f, acc in fold_results.items():
    print(f"{f} : {acc:.4f}")
print(f"RATA-RATA AKURASI K-FOLD: {sum(fold_results.values())/K_FOLDS:.4f}")
print("="*40)


MEMULAI FOLD 1/5


Fold 1 - Epoch 1/12 [Train]:   0%|          | 0/72 [00:00<?, ?it/s]

Fold 1 | Epoch 1 | Val Acc: 0.8028 | Val Loss: 0.6391


Fold 1 - Epoch 2/12 [Train]:   0%|          | 0/72 [00:00<?, ?it/s]

Fold 1 | Epoch 2 | Val Acc: 0.7024 | Val Loss: 0.8216


...

Fold 5 | Epoch 10 | Val Acc: 0.9201 | Val Loss: 0.4312


Fold 5 - Epoch 11/12 [Train]:   0%|          | 0/73 [00:00<?, ?it/s]

Fold 5 | Epoch 11 | Val Acc: 0.9201 | Val Loss: 0.4391


Fold 5 - Epoch 12/12 [Train]:   0%|          | 0/73 [00:00<?, ?it/s]

Fold 5 | Epoch 12 | Val Acc: 0.9201 | Val Loss: 0.4405
Fold 5 Selesai! Model terbaik disimpan di ../Models/convnext_fold_5.pth dengan Akurasi: 0.9201

SEMUA FOLD SELESAI DILATIH!
Ringkasan Akurasi Validasi:
Fold 1 : 0.8824
Fold 2 : 0.8993
Fold 3 : 0.9132
Fold 4 : 0.9375
Fold 5 : 0.9201
RATA-RATA AKURASI K-FOLD: 0.9105
